# 09 SwiGLU 如何实现，和 ReLU FFN 有何取舍？

## 面试回答主线

SwiGLU 是门控前馈层：一条分支产生 value，另一条分支先做 SiLU 再充当 gate，二者逐元素相乘后投影回模型维度。相较单路 ReLU FFN，它多了一组投影参数，却让特征选择具有输入依赖的连续门控。面试时应写清 $\operatorname{SiLU}(a)=a\sigma(a)$，并说明为了参数预算公平，SwiGLU 的中间维度常会相应缩小。本实验手写 ReLU FFN 与 SwiGLU 分类器，观察训练 loss、准确率和 gate 激活；随后演示漏 SiLU 时尺度过大的失败。小数据只解释门控机制。

**核心公式：** $\operatorname{SwiGLU}(x)=(xW_g\odot\sigma(xW_g))\odot(xW_v)$，然后经 $W_o$ 投影。实现中常写为 $\operatorname{SiLU}(xW_g)\odot(xW_v)$。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
class ReLUFFN(nn.Module):  # 手写两层 ReLU 前馈分类器作为基线。
    def __init__(self, width=4):  # 接收中间维度。
        super().__init__()  # 初始化模块父类。
        self.w_in = nn.Parameter(torch.randn(3, width) * 0.2)  # 创建输入投影矩阵。
        self.w_out = nn.Parameter(torch.randn(width, 2) * 0.2)  # 创建输出投影矩阵。
    def forward(self, batch):  # 显式实现 ReLU FFN 前向传播。
        hidden = torch.relu(batch @ self.w_in)  # 计算单路 ReLU 隐层。
        return hidden @ self.w_out  # 生成分类 logits。
def manual_train(model, steps=80):  # 不调用 Optimizer，手写梯度下降训练。
    losses = []  # 保存损失曲线。
    for _ in range(steps):  # 执行固定数量的更新。
        logits = model(features)  # 计算当前 logits。
        loss = torch.nn.functional.cross_entropy(logits, labels)  # 计算分类交叉熵。
        gradients = torch.autograd.grad(loss, list(model.parameters()))  # 获取各参数梯度。
        with torch.no_grad():  # 在不构图环境中写入参数更新。
            for parameter, gradient in zip(model.parameters(), gradients):  # 配对遍历参数与梯度。
                parameter -= 0.35 * gradient  # 执行手写 SGD 更新。
        losses.append(float(loss))  # 保存当前损失。
    accuracy = float(model(features).argmax(dim=1).eq(labels).float().mean())  # 计算训练后准确率。
    return losses, accuracy  # 返回曲线与准确率。
relu_model = ReLUFFN()  # 创建 ReLU 基线模型。
relu_losses, relu_accuracy = manual_train(relu_model)  # 训练基线模型。
baseline_metric = relu_accuracy  # 保存基线准确率。
print(f'ReLU FFN：loss={relu_losses[0]:.4f}->{relu_losses[-1]:.4f}，准确率={relu_accuracy:.2f}')  # 展示基线训练结果。


ReLU FFN：loss=0.7027->0.0218，准确率=1.00


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
class SwiGLUFFN(nn.Module):  # 手写带连续门控的 SwiGLU 前馈分类器。
    def __init__(self, width=3):  # 用较小中间维度近似控制参数预算。
        super().__init__()  # 初始化模块父类。
        self.w_gate = nn.Parameter(torch.randn(3, width) * 0.2)  # 创建 gate 投影矩阵。
        self.w_value = nn.Parameter(torch.randn(3, width) * 0.2)  # 创建 value 投影矩阵。
        self.w_out = nn.Parameter(torch.randn(width, 2) * 0.2)  # 创建输出投影矩阵。
    def forward(self, batch):  # 明确实现 SiLU 门控和输出投影。
        gate_input = batch @ self.w_gate  # 计算 gate 的线性输入。
        gate = gate_input * torch.sigmoid(gate_input)  # 手写 SiLU 激活而非隐藏在高层网络中。
        value = batch @ self.w_value  # 计算 value 分支。
        return (gate * value) @ self.w_out  # 按元素门控后输出分类 logits。
swiglu_model = SwiGLUFFN()  # 创建 SwiGLU 核心模型。
swiglu_losses, swiglu_accuracy = manual_train(swiglu_model)  # 使用相同手写训练循环训练门控模型。
sample_gate_input = features @ swiglu_model.w_gate  # 取出六条工单的 gate 输入。
sample_gate = sample_gate_input * torch.sigmoid(sample_gate_input)  # 计算可解释的 gate 激活。
core_metric = swiglu_accuracy  # 保存核心准确率。
print(f'SwiGLU：loss={swiglu_losses[0]:.4f}->{swiglu_losses[-1]:.4f}，准确率={swiglu_accuracy:.2f}，gate 均值={float(sample_gate.mean()):.4f}')  # 输出门控中间量。


SwiGLU：loss=0.6888->0.0571，准确率=1.00，gate 均值=0.1332


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=1.000000
核心机制     | 指标=1.000000


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **SwiGLU** 的关键状态与更新路径。SwiGLU 的两条输入投影提高参数和带宽成本；比较时应固定总参数/FLOPs，而不是给门控模型更多中间维度。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
oversized_gate_input = 12.0 * sample_gate_input  # 模拟异常大激活进入门控分支。
broken_gate = oversized_gate_input * (features @ swiglu_model.w_value)  # 故意省略 SiLU，直接把两条大分支相乘。
failure_metric = float(broken_gate.pow(2).mean().sqrt())  # 测量错误裸乘门控的 RMS。
fixed_gate = (oversized_gate_input * torch.sigmoid(oversized_gate_input)) * (features @ swiglu_model.w_value)  # 恢复 SiLU 后再门控。
fix_metric = float(fixed_gate.pow(2).mean().sqrt())  # 测量修复门控的 RMS。
print(f'失败：裸乘 gate RMS={failure_metric:.3f}；修复：SiLU gate RMS={fix_metric:.3f}')  # 展示激活函数不是可省略细节。


失败：裸乘 gate RMS=21.844；修复：SiLU gate RMS=17.161


## 工程取舍、常见坑与延伸追问

**工程取舍：** SwiGLU 的两条输入投影提高参数和带宽成本；比较时应固定总参数/FLOPs，而不是给门控模型更多中间维度。

**常见坑：** 把 gate 写成裸乘法、忽略中间维度预算，或把训练集上的微小准确率差当成架构定论。

**延伸追问：** 为何 GLU 家族常用更小的 hidden width 才能与 ReLU FFN 参数公平？门控饱和时如何从激活分布和梯度诊断？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert 0.0 <= baseline_metric <= 1.0  # 验证 ReLU 基线准确率合法。
assert 0.0 <= core_metric <= 1.0  # 验证 SwiGLU 准确率合法。
assert torch.isfinite(sample_gate).all()  # 验证门控激活没有数值异常。
assert failure_metric > fix_metric  # 验证 SiLU 限制了本异常输入的门控尺度。
